In [ ]:
# Set the model name
model_name = "lvwerra/gpt2-imdb-pos-v2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create a text generation pipeline
text_generator = pipeline('text-generation', model=model, tokenizer=tokenizer)

review_prompt = "Surprisingly, the film"

# Generate a continuation of the review
generated_text = text_generator(review_prompt, max_length=10)
print(f"Generated Review Continuation: {generated_text[0]['generated_text']}")

In [ ]:
# Create a sentiment analysis pipeline
sentiment_analyzer = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

review_text = "Surprisingly, the film is a very good one"

# Classify the sentiment of the review
sentiment = sentiment_analyzer(review_text)
print(f"Sentiment Analysis Result: {sentiment}")

In [ ]:
dataset = load_dataset("argilla/tripadvisor-hotel-reviews")

tokenizer = AutoTokenizer.from_pretrained("openai-gpt")

# Add padding with the pad token
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

def tokenize_function(examples):
   return tokenizer(examples["text"], padding="max_length", truncation=True)

# Tokenize the dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

In [ ]:
# Define the train and test datasets
training_dataset = tokenized_datasets["train"]
testing_dataset = tokenized_datasets["test"]

# Initialize the trainer class
trainer = Trainer(
# Add arguments to the class
    model=model,
    args=training_args,
    train_dataset=training_dataset,
    eval_dataset=testing_dataset
)

In [ ]:
# Load the dataset
preference_data = load_dataset("trl-internal-testing/hh-rlhf-helpful-base-trl-style", split="train")

# Define a function to extract the prompt
def extract_prompt(text):
    prompt = text[0]['content']
    return prompt

# Apply the function to the dataset 
preference_data_with_prompt = preference_data.map(
    lambda sample: {**sample, 'prompt': extract_prompt(sample['chosen'])}
)

sample = preference_data_with_prompt.select(range(1))
print(sample['prompt'])

In [ ]:
def evaluate_slogans(slogans_X, slogans_Y):
    wins_X, wins_Y = 0, 0
    for (slogan_X, score_X), (slogan_Y, score_Y) in zip(slogans_X, slogans_Y):
        # Assign one point to X if score X is higher, otherwise to Y
        if score_X > score_Y:
            wins_X += 1
        else:
            wins_Y += 1
    success_rate_X = (wins_X / len(slogans_X)) * 100
    success_rate_Y = (wins_Y / len(slogans_Y)) * 100
    return success_rate_X, success_rate_Y

results = evaluate_slogans(slogans_X, slogans_Y)
print(f"The resulting scores are {results}")

In [ ]:
# Define the filter function
def filter_low_confidence_predictions(prob_dists, threshold=0.5):
    filtered_indices = [i for i, prob_dist in enumerate(prob_dists) if least_confidence(prob_dist) > threshold]
    return filtered_indices

# Find the indices
filtered_indices = filter_low_confidence_predictions(prob_dists)

high_confidence_texts = [texts[i] for i in filtered_indices]
print("High-confidence texts:", high_confidence_texts)

In [ ]:
def detect_anomalies(data, n_clusters=3):
    # Initialize k-means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    clusters = kmeans.fit_predict(data)
    centers = kmeans.cluster_centers_

    # Calculate distances from cluster centers
    distances = np.linalg.norm(data - centers[clusters], axis=1)
    return distances
  
anomalies = detect_anomalies(confidences)
print(anomalies)

In [ ]:
# Create the active learner object
learner = ActiveLearner(
    # Set the estimator 
    estimator=LogisticRegression(),
    # Set the query strategy
    query_strategy=uncertainty_sampling,
    # Pass the labeled data
    X_training=X_labeled, y_training=y_labeled
)

In [ ]:
# Set the number of queries
n_queries = 10
for _ in range(n_queries):
    # Use the current labeled data
    learner.teach(X_labeled, y_labeled)
    # Query from unlabeled data
    query_idx, _ = learner.query(X_unlabeled, n_instances=5)  
    X_new, y_new = X_unlabeled[query_idx], y[query_idx]  
    X_labeled = np.vstack((X_labeled, X_new))  
    y_labeled = np.append(y_labeled, y_new)  
    # Update the unlabeled dataset
    X_unlabeled = np.delete(X_unlabeled, query_idx, axis=0) 

In [ ]:
# Load the pre-trained GPT-1 model for text classification
model = AutoModelForSequenceClassification.from_pretrained('openai-gpt')

tokenizer = AutoTokenizer.from_pretrained("openai-gpt")

# Initialize the reward configuration and set max_length
config = RewardConfig(output_dir='output_dir', max_length=60)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("openai-gpt")
model = AutoModelForSequenceClassification.from_pretrained('openai-gpt')
config = RewardConfig(output_dir='output_dir', max_length=60)

# Initialize the reward trainer
trainer = RewardTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=config
)

In [ ]:
from trl import PPOConfig, AutoModelForCausalLMWithValueHead, PPOTrainer
from transformers import AutoTokenizer

# Initialize PPO Configuration
gpt2_config = PPOConfig(model_name="gpt2", learning_rate=1.2e-5)

# Load the model
gpt2_model = AutoModelForCausalLMWithValueHead.from_pretrained(gpt2_config.model_name)
gpt2_tokenizer = AutoTokenizer.from_pretrained(gpt2_config.model_name)

# Initialize PPO Trainer
ppo_trainer = PPOTrainer(model=gpt2_model, config=gpt2_config, dataset=dataset_cs, tokenizer=gpt2_tokenizer)

In [ ]:
for batch in tqdm(ppo_trainer.dataloader): 

    # Generate responses for the given queries using the trainer
    response_tensors = ppo_trainer.generate(batch["input_ids"])

    batch["response"] = [tokenizer.decode(r.squeeze()) for r in response_tensors]

    texts = [q + r for q, r in zip(batch["query"], batch["response"])]

    rewards = reward_model(texts)

    # Training PPO step with the query, responses ids, and rewards
    stats = ppo_trainer.step(batch["input_ids"], response_tensors, rewards)

    ppo_trainer.log_stats(stats, batch, rewards)

In [ ]:
model_name = "gpt2"  

# Load the model in 8-bit precision
pretrained_model = AutoModelForCausalLM.from_pretrained(
                                                       model_name, 
                                                       load_in_8bit=True
                                                      )

# Prepare the model for fine-tuning
pretrained_model_8bit = prepare_model_for_int8_training(pretrained_model)

# Load the model with a value head
model = AutoModelForCausalLMWithValueHead.from_pretrained(pretrained_model_8bit)

In [ ]:
# Set the configuration parameters
config = LoraConfig(
    r=32,  
    lora_alpha=32,  
    lora_dropout=0.1,  
    bias="lora_only")  

# Apply the LoRA configuration to the 8-bit model
lora_model = get_peft_model(pretrained_model_8bit, config)
# Set up the tokenizer and model with a value head for PPO training
model = AutoModelForCausalLMWithValueHead.from_pretrained(lora_model)

In [ ]:
generation_kwargs = {
    # Set min length and top k parameters
    "min_length": -1,
  	"top_k": 0.0, 
  	"top_p": 1.0,
  	"do_sample": True,  
  	"pad_token_id": tokenizer.eos_token_id, 
  	"max_new_tokens": 32}

In [ ]:
def majority_vote(df):
	# Count occurrences of each (chosen, rejected) pair
    votes = Counter(zip(df['chosen'], df['rejected']))
    # Find the (chosen, rejected) pair with the highest vote count
    winner = max(votes, key=votes.get)
    return winner

final_preferences = quality_df.groupby(['id']).apply(majority_vote)

print(final_preferences)

In [ ]:
def detect_unreliable_source(df):
    df_majority = df.groupby('id').apply(majority_vote)
    disagreements = {source: 0 for source in df['source'].unique()}
    for _, row in df.iterrows():
        # Condition to find a disagreement with majority vote
        if (row['chosen'], row['rejected']) != df_majority[row['id']]:
            disagreements[row['source']] += 1
    unreliable_source = max(disagreements, key=disagreements.get)
    return unreliable_source

disagreement = detect_unreliable_source(automotive_df)
print("Unreliable Source:", disagreement)